# Phase 5 — Feature Engineering (DuckDB + Pandas)

**Input:**  `D:/project/data/processed/bts_weather_joined.parquet`  
**Output:** `D:/project/data/processed/features_final.parquet`  
**Engine:** DuckDB — no Spark needed, runs in seconds

### Note on 2020 Data
2020 is **included** with explicit flags. COVID caused ~60% flight volume drop and paradoxically lower delay rates due to empty aircraft and uncongested airspace. We handle this by:
- `is_covid_year = 1` for all 2020 rows
- `flight_volume_index` captures traffic collapse continuously
- Airport/carrier historical baselines exclude 2020 so they reflect true normal behavior

### Feature Groups
| Group | Features |
|---|---|
| Time | hour, day-of-week, month, quarter, season, is_weekend, is_holiday_period |
| Weather | severity flags, wind/visibility/precip categories, composite score |
| Airport | hub flag, historical delay rate, daily congestion index |
| Route | distance bucket, route frequency, carrier delay rate |
| Anomaly | is_covid_year, flight_volume_index |

---
## 0. Setup

In [1]:
# !pip install duckdb pandas pyarrow

import duckdb
import pandas as pd
import os

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

PARQUET_IN  = "D:/project/data/processed/bts_weather_joined.parquet"
PARQUET_OUT = "D:/project/data/processed/features_final.parquet"
DB_PATH     = "D:/project/data/processed/flight_delay.duckdb"

os.makedirs("D:/project/data/processed", exist_ok=True)

HUB_AIRPORTS = [
    "ATL", "LAX", "ORD", "DFW", "DEN",
    "JFK", "SFO", "CLT", "LAS", "PHX",
    "MIA", "IAH", "SEA", "EWR", "BOS"
]

print("Config ready.")

Config ready.


---
## 1. Connect DuckDB & Load Parquet

In [2]:
con = duckdb.connect(DB_PATH)
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=4")

# Register the parquet file as a view
con.execute(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM read_parquet('{PARQUET_IN}')")

# Quick check
con.execute("""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT AIRPORT_CODE) AS airports,
           MIN(FL_DATE) AS from_date, MAX(FL_DATE) AS to_date
    FROM raw
""").df()

,rows,airports,from_date,to_date
0,48523795,352,2015-01-01,2024-12-31


---
## 2. Time Features

In [3]:
con.execute("""
CREATE OR REPLACE VIEW time_features AS
SELECT *,
    -- Calendar extracts
    YEAR(CAST(FL_DATE AS DATE))                     AS year,
    MONTH(CAST(FL_DATE AS DATE))                    AS month,
    DAY(CAST(FL_DATE AS DATE))                      AS day_of_month,
    DAYOFWEEK(CAST(FL_DATE AS DATE))                AS day_of_week,   -- 0=Sun, 6=Sat
    WEEKOFYEAR(CAST(FL_DATE AS DATE))               AS week_of_year,
    QUARTER(CAST(FL_DATE AS DATE))                  AS quarter,

    -- Weekend flag (0=Sun, 6=Sat)
    CASE WHEN DAYOFWEEK(CAST(FL_DATE AS DATE)) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend,

    -- Season
    CASE
        WHEN MONTH(CAST(FL_DATE AS DATE)) IN (12,1,2) THEN 'Winter'
        WHEN MONTH(CAST(FL_DATE AS DATE)) IN (3,4,5)  THEN 'Spring'
        WHEN MONTH(CAST(FL_DATE AS DATE)) IN (6,7,8)  THEN 'Summer'
        ELSE 'Fall'
    END AS season,

    -- Departure hour from CRSDepTime (stored as HHMM integer e.g. 1430 = 14)
    CASE
        WHEN CRSDepTime IS NOT NULL
        THEN CAST(CAST(CRSDepTime AS INTEGER) / 100 AS INTEGER)
        ELSE NULL
    END AS dep_hour,

    -- Time of day bucket
    CASE
        WHEN CAST(CAST(CRSDepTime AS INTEGER) / 100 AS INTEGER) BETWEEN 5  AND 11 THEN 'Morning'
        WHEN CAST(CAST(CRSDepTime AS INTEGER) / 100 AS INTEGER) BETWEEN 12 AND 16 THEN 'Afternoon'
        WHEN CAST(CAST(CRSDepTime AS INTEGER) / 100 AS INTEGER) BETWEEN 17 AND 20 THEN 'Evening'
        WHEN CAST(CAST(CRSDepTime AS INTEGER) / 100 AS INTEGER) BETWEEN 21 AND 23 THEN 'Night'
        ELSE 'RedEye'
    END AS time_of_day,

    -- Holiday period flag
    CASE WHEN (
        (MONTH(CAST(FL_DATE AS DATE)) = 1  AND DAY(CAST(FL_DATE AS DATE)) BETWEEN 1  AND 2)  OR  -- New Year
        (MONTH(CAST(FL_DATE AS DATE)) = 5  AND DAY(CAST(FL_DATE AS DATE)) BETWEEN 23 AND 27) OR  -- Memorial Day
        (MONTH(CAST(FL_DATE AS DATE)) = 7  AND DAY(CAST(FL_DATE AS DATE)) BETWEEN 1  AND 7)  OR  -- July 4th
        (MONTH(CAST(FL_DATE AS DATE)) = 9  AND DAY(CAST(FL_DATE AS DATE)) BETWEEN 1  AND 2)  OR  -- Labor Day
        (MONTH(CAST(FL_DATE AS DATE)) = 11 AND DAY(CAST(FL_DATE AS DATE)) BETWEEN 20 AND 30) OR  -- Thanksgiving
        (MONTH(CAST(FL_DATE AS DATE)) = 12 AND DAY(CAST(FL_DATE AS DATE)) BETWEEN 20 AND 31)     -- Christmas
    ) THEN 1 ELSE 0 END AS is_holiday_period

FROM raw
""")

# Verify
con.execute("""
    SELECT FL_DATE, year, month, day_of_week, season,
           dep_hour, time_of_day, is_weekend, is_holiday_period
    FROM time_features LIMIT 5
""").df()

,FL_DATE,year,month,day_of_week,season,dep_hour,time_of_day,is_weekend,is_holiday_period
0,2016-01-09,2016,1,6,Winter,6,Morning,1,0
1,2016-01-12,2016,1,2,Winter,15,Afternoon,0,0
2,2016-01-16,2016,1,6,Winter,11,Morning,1,0
3,2016-01-18,2016,1,1,Winter,15,Afternoon,0,0
4,2016-01-19,2016,1,2,Winter,15,Afternoon,0,0


---
## 3. COVID / Anomaly Features

**2020 note:** Delay rates dropped paradoxically during COVID due to near-empty aircraft and uncongested airspace — not because conditions improved. The `is_covid_year` flag and `flight_volume_index` let models learn this regime separately rather than treating it as normal low-delay behavior.

In [4]:
# Monthly flight volume per airport, normalized 0-1 within each airport
monthly_vol = con.execute("""
    SELECT
        AIRPORT_CODE,
        YEAR(CAST(FL_DATE AS DATE))  AS year,
        MONTH(CAST(FL_DATE AS DATE)) AS month,
        COUNT(*) AS monthly_flights
    FROM raw
    GROUP BY AIRPORT_CODE, YEAR(CAST(FL_DATE AS DATE)), MONTH(CAST(FL_DATE AS DATE))
""").df()

# Normalize per airport
monthly_vol["flight_volume_index"] = monthly_vol.groupby("AIRPORT_CODE")["monthly_flights"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
).round(4)

# Register back as DuckDB table
con.register("monthly_vol_tbl", monthly_vol)

con.execute("""
CREATE OR REPLACE VIEW covid_features AS
SELECT
    t.*,
    CASE WHEN t.year = 2020 THEN 1 ELSE 0 END AS is_covid_year,
    m.monthly_flights,
    m.flight_volume_index
FROM time_features t
LEFT JOIN monthly_vol_tbl m
    ON t.AIRPORT_CODE = m.AIRPORT_CODE
    AND YEAR(CAST(t.FL_DATE AS DATE))  = m.year
    AND MONTH(CAST(t.FL_DATE AS DATE)) = m.month
""")

# Verify — 2020 should show lowest flight_volume_index
con.execute("""
    SELECT year,
           COUNT(*) AS flights,
           ROUND(AVG(flight_volume_index), 3) AS avg_vol_index,
           MAX(is_covid_year) AS is_covid
    FROM covid_features
    GROUP BY year ORDER BY year
""").df()

,year,flights,avg_vol_index,is_covid
0,2015,4680502,0.6510,0
1,2016,4447929,0.6120,0
2,2017,4456437,0.6160,0
3,2018,5469982,0.7880,0
4,2019,5601002,0.8110,0
5,2020,3570635,0.5000,1
6,2021,4641693,0.6450,0
7,2022,5097112,0.7050,0
8,2023,5184157,0.7310,0
9,2024,5374346,0.7700,0


---
## 4. Weather Features

In [5]:
con.execute("""
CREATE OR REPLACE VIEW weather_features AS
SELECT *,

    -- Temperature flags
    CASE WHEN min_temp_c < 0    THEN 1 ELSE 0 END AS is_freezing,
    CASE WHEN max_temp_c > 38   THEN 1 ELSE 0 END AS is_extreme_heat,
    ROUND(max_temp_c - min_temp_c, 2)             AS temp_range_c,

    -- Wind categories (m/s)
    CASE
        WHEN max_wind_ms < 5  THEN 'Calm'
        WHEN max_wind_ms < 10 THEN 'Moderate'
        WHEN max_wind_ms < 15 THEN 'Strong'
        WHEN max_wind_ms < 20 THEN 'Very Strong'
        ELSE 'Severe'
    END AS wind_category,
    CASE WHEN max_wind_ms >= 15 THEN 1 ELSE 0 END AS is_high_wind,

    -- Visibility categories (km) — IFR/VFR flight rules
    CASE
        WHEN min_visibility_km < 1.6 THEN 'LIFR'
        WHEN min_visibility_km < 5.0 THEN 'IFR'
        WHEN min_visibility_km < 8.0 THEN 'MVFR'
        ELSE 'VFR'
    END AS visibility_category,
    CASE WHEN min_visibility_km < 5.0 THEN 1 ELSE 0 END AS is_low_visibility,

    -- Precipitation categories (mm)
    CASE
        WHEN total_precip_mm IS NULL OR total_precip_mm = 0 THEN 'None'
        WHEN total_precip_mm < 2.5  THEN 'Light'
        WHEN total_precip_mm < 10.0 THEN 'Moderate'
        WHEN total_precip_mm < 50.0 THEN 'Heavy'
        ELSE 'Extreme'
    END AS precip_category,
    CASE WHEN total_precip_mm > 0 THEN 1 ELSE 0 END AS is_precipitation,

    -- Ceiling flag (< 300m = below 1000ft = IFR conditions)
    CASE WHEN min_ceiling_m < 300 THEN 1 ELSE 0 END AS is_low_ceiling,

    -- Composite weather severity score (0-5)
    (
        CASE WHEN min_temp_c < 0        THEN 1 ELSE 0 END +
        CASE WHEN max_temp_c > 38       THEN 1 ELSE 0 END +
        CASE WHEN max_wind_ms >= 15     THEN 1 ELSE 0 END +
        CASE WHEN min_visibility_km < 5 THEN 1 ELSE 0 END +
        CASE WHEN min_ceiling_m < 300   THEN 1 ELSE 0 END
    ) AS weather_severity_score

FROM covid_features
""")

# Verify
con.execute("""
    SELECT AIRPORT_CODE, FL_DATE,
           is_freezing, is_high_wind, wind_category,
           visibility_category, precip_category,
           weather_severity_score
    FROM weather_features LIMIT 5
""").df()

,AIRPORT_CODE,FL_DATE,is_freezing,is_high_wind,wind_category,visibility_category,precip_category,weather_severity_score
0,ATL,2016-01-05,1,0,Moderate,VFR,Moderate,1
1,ATL,2016-01-17,0,0,Moderate,MVFR,Moderate,1
2,ATL,2016-01-21,0,0,Moderate,IFR,Moderate,2
3,ATL,2016-01-27,0,0,Moderate,LIFR,Moderate,2
4,ATL,2016-02-04,0,0,Moderate,VFR,Moderate,0


---
## 5. Airport Features

In [6]:
# Hub flag
hub_list = "','" .join(HUB_AIRPORTS)

# Historical airport delay rate — excludes 2020
airport_stats = con.execute(f"""
    SELECT
        AIRPORT_CODE,
        ROUND(AVG(CAST(DEP_DEL15 AS FLOAT)), 4)    AS airport_hist_delay_rate,
        ROUND(AVG(DepDelayMinutes), 2)              AS airport_hist_avg_delay_mins
    FROM weather_features
    WHERE year != 2020
    AND DEP_DEL15 IS NOT NULL
    GROUP BY AIRPORT_CODE
""").df()

con.register("airport_stats_tbl", airport_stats)

# Daily congestion index — flights today vs avg for that airport+month
daily_traffic = con.execute("""
    SELECT AIRPORT_CODE, FL_DATE,
           COUNT(*) AS daily_flight_count
    FROM weather_features
    GROUP BY AIRPORT_CODE, FL_DATE
""").df()

monthly_avg = con.execute("""
    SELECT AIRPORT_CODE,
           MONTH(CAST(FL_DATE AS DATE)) AS month,
           AVG(cnt) AS avg_daily_flights
    FROM (
        SELECT AIRPORT_CODE, FL_DATE,
               MONTH(CAST(FL_DATE AS DATE)) AS month,
               COUNT(*) AS cnt
        FROM weather_features
        GROUP BY AIRPORT_CODE, FL_DATE, MONTH(CAST(FL_DATE AS DATE))
    )
    GROUP BY AIRPORT_CODE, MONTH(CAST(FL_DATE AS DATE))
""").df()

# Merge in pandas
daily_traffic["month"] = pd.to_datetime(daily_traffic["FL_DATE"]).dt.month
daily_traffic = daily_traffic.merge(monthly_avg, on=["AIRPORT_CODE", "month"], how="left")
daily_traffic["congestion_index"] = (
    daily_traffic["daily_flight_count"] / (daily_traffic["avg_daily_flights"] + 1e-9)
).round(4)
daily_traffic = daily_traffic[["AIRPORT_CODE", "FL_DATE", "daily_flight_count", "congestion_index"]]

con.register("daily_traffic_tbl", daily_traffic)

con.execute(f"""
CREATE OR REPLACE VIEW airport_features AS
SELECT
    w.*,
    CASE WHEN w.AIRPORT_CODE IN ('{hub_list}') THEN 1 ELSE 0 END AS is_hub,
    a.airport_hist_delay_rate,
    a.airport_hist_avg_delay_mins,
    d.daily_flight_count,
    d.congestion_index
FROM weather_features w
LEFT JOIN airport_stats_tbl a ON w.AIRPORT_CODE = a.AIRPORT_CODE
LEFT JOIN daily_traffic_tbl d ON w.AIRPORT_CODE = d.AIRPORT_CODE AND w.FL_DATE = d.FL_DATE
""")

print("Airport features added.")
con.execute("""
    SELECT AIRPORT_CODE, is_hub,
           ROUND(airport_hist_delay_rate*100,1) AS hist_delay_pct,
           ROUND(congestion_index,3) AS congestion
    FROM airport_features
    GROUP BY AIRPORT_CODE, is_hub, airport_hist_delay_rate, congestion_index
    LIMIT 10
""").df()

Airport features added.


,AIRPORT_CODE,is_hub,hist_delay_pct,congestion
0,ATL,1,17.7000,1.1530
1,BOS,1,19.7000,1.0800
2,BOS,1,19.7000,1.0830
3,CLT,1,19.0000,0.5900
4,CLT,1,19.0000,1.2230
5,CLT,1,19.0000,1.1920
6,CLT,1,19.0000,1.3990
7,CLT,1,19.0000,1.4380
8,DEN,1,22.6000,0.7920
9,DEN,1,22.6000,0.9730


In [7]:
# Show historical delay rate ranking
airport_stats.sort_values("airport_hist_delay_rate", ascending=False)

,AIRPORT_CODE,airport_hist_delay_rate,airport_hist_avg_delay_mins
250,MGW,0.5263,87.2100
85,SWF,0.4105,29.4000
333,SCK,0.3273,24.0300
187,BLV,0.3015,25.1500
349,STC,0.3012,23.4100
...,...,...,...
124,BTM,0.0654,8.9300
319,PIH,0.0621,6.2900
31,EKO,0.0579,9.5800
169,GUM,0.0000,0.8200


---
## 6. Route & Carrier Features

In [8]:
# Route frequency
route_freq = con.execute("""
    SELECT AIRPORT_CODE, Dest, COUNT(*) AS route_total_flights
    FROM airport_features
    GROUP BY AIRPORT_CODE, Dest
""").df()

# Carrier historical delay rate (excluding 2020)
carrier_stats = con.execute("""
    SELECT
        Reporting_Airline,
        ROUND(AVG(CAST(DEP_DEL15 AS FLOAT)), 4) AS carrier_hist_delay_rate,
        COUNT(*) AS carrier_total_flights
    FROM airport_features
    WHERE year != 2020 AND DEP_DEL15 IS NOT NULL
    GROUP BY Reporting_Airline
""").df()

con.register("route_freq_tbl", route_freq)
con.register("carrier_stats_tbl", carrier_stats)

con.execute("""
CREATE OR REPLACE VIEW route_features AS
SELECT
    a.*,
    -- Distance bucket
    CASE
        WHEN Distance < 500  THEN 'Short'
        WHEN Distance < 1500 THEN 'Medium'
        WHEN Distance < 2500 THEN 'Long'
        ELSE 'Ultra'
    END AS distance_bucket,
    r.route_total_flights,
    c.carrier_hist_delay_rate,
    c.carrier_total_flights
FROM airport_features a
LEFT JOIN route_freq_tbl   r ON a.AIRPORT_CODE = r.AIRPORT_CODE AND a.Dest = r.Dest
LEFT JOIN carrier_stats_tbl c ON a.Reporting_Airline = c.Reporting_Airline
""")

print("Route and carrier features added.")
print("\nCarrier delay rate ranking:")
carrier_stats.sort_values("carrier_hist_delay_rate", ascending=False)

Route and carrier features added.

Carrier delay rate ranking:


,Reporting_Airline,carrier_hist_delay_rate,carrier_total_flights
0,B6,0.2634,1840762
7,F9,0.2586,914171
4,WN,0.2393,6578464
5,NK,0.2272,1188055
19,G4,0.2198,155699
8,VX,0.2127,206993
10,AA,0.1949,7544930
11,UA,0.1900,5299371
17,YV,0.1879,612268
16,HA,0.1724,119808


---
## 7. Final Dataset — Select & Save

In [9]:
# Class balance check before saving
con.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN DEP_DEL15 = 1 THEN 1 ELSE 0 END) AS delayed,
        SUM(CASE WHEN DEP_DEL15 = 0 THEN 1 ELSE 0 END) AS on_time,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 1) AS delay_rate_pct
    FROM route_features
    WHERE DEP_DEL15 IS NOT NULL
""").df()

,total,delayed,on_time,delay_rate_pct
0,48523795,8739593.0000,39784202.0000,18.0000


In [10]:
# Write final Parquet — select clean ordered columns
con.execute(f"""
COPY (
    SELECT
        -- IDs
        FL_DATE, AIRPORT_CODE, Dest, Reporting_Airline,

        -- Target
        DEP_DEL15,

        -- Raw delay (reference only — not model inputs)
        DepDelay, DepDelayMinutes, ArrDelay, ArrDelayMinutes, ARR_DEL15,
        CarrierDelay, WeatherDelay, NASDelay, SecurityDelay, LateAircraftDelay,
        Cancelled, CancellationCode,

        -- Time features
        year, month, day_of_month, day_of_week, week_of_year, quarter,
        dep_hour, time_of_day, season, is_weekend, is_holiday_period,

        -- Anomaly features
        is_covid_year, monthly_flights, flight_volume_index,

        -- Raw weather
        avg_temp_c, min_temp_c, max_temp_c, avg_dew_point_c,
        avg_visibility_km, min_visibility_km,
        avg_wind_ms, max_wind_ms,
        total_precip_mm, max_precip_mm,
        avg_pressure_hpa, avg_ceiling_m, min_ceiling_m,

        -- Engineered weather
        is_freezing, is_extreme_heat, temp_range_c,
        wind_category, is_high_wind,
        visibility_category, is_low_visibility,
        precip_category, is_precipitation,
        is_low_ceiling, weather_severity_score,

        -- Airport features
        is_hub, airport_hist_delay_rate, airport_hist_avg_delay_mins,
        daily_flight_count, congestion_index,

        -- Route features
        Distance, distance_bucket, route_total_flights,
        carrier_hist_delay_rate, carrier_total_flights

    FROM route_features
    WHERE DEP_DEL15 IS NOT NULL
)
TO '{PARQUET_OUT}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Saved to: {PARQUET_OUT}")

Saved to: D:/project/data/processed/features_final.parquet


In [11]:
# Verify
result = con.execute(f"""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT AIRPORT_CODE) AS airports
    FROM read_parquet('{PARQUET_OUT}')
""").df()

size_mb = os.path.getsize(PARQUET_OUT) / 1e6

print(result.to_string(index=False))
print(f"File size : {size_mb:.1f} MB")

    rows  airports
48523795       352
File size : 731.7 MB


In [12]:
# Final summary
stats = con.execute(f"""
    SELECT
        COUNT(*) AS total_flights,
        COUNT(DISTINCT AIRPORT_CODE) AS airports,
        MIN(FL_DATE) AS date_from,
        MAX(FL_DATE) AS date_to,
        ROUND(100.0 * AVG(CAST(DEP_DEL15 AS FLOAT)), 1) AS delay_rate_pct
    FROM read_parquet('{PARQUET_OUT}')
""").df()

print("=" * 55)
print("  PHASE 5 COMPLETE")
print("=" * 55)
for col in stats.columns:
    print(f"  {col:<28} : {stats[col].iloc[0]}")
print(f"  {'File size':<28} : {size_mb:.1f} MB")
print(f"  {'Output':<28} : {PARQUET_OUT}")
print("")
print("  2020 handling:")
print("  - Included, flagged via is_covid_year=1")
print("  - flight_volume_index captures traffic collapse")
print("  - Airport/carrier baselines exclude 2020")
print("")
print("  Next → Phase 6: Model Training")
print("  - Logistic Regression (baseline)")
print("  - Random Forest")
print("  - Gradient Boosted Trees")
print("  - Evaluate: AUC-ROC, F1, Precision, Recall")
print("=" * 55)

con.close()

  PHASE 5 COMPLETE
  total_flights                : 48523795
  airports                     : 352
  date_from                    : 2015-01-01 00:00:00
  date_to                      : 2024-12-31 00:00:00
  delay_rate_pct               : 18.0
  File size                    : 731.7 MB
  Output                       : D:/project/data/processed/features_final.parquet

  2020 handling:
  - Included, flagged via is_covid_year=1
  - flight_volume_index captures traffic collapse
  - Airport/carrier baselines exclude 2020

  Next → Phase 6: Model Training
  - Logistic Regression (baseline)
  - Random Forest
  - Gradient Boosted Trees
  - Evaluate: AUC-ROC, F1, Precision, Recall
